In [1]:
from einops import rearrange
import torch

rearrange(torch.randn(10, 5), 'n d -> n 1 d').shape

torch.Size([10, 1, 5])

In [4]:
from sortedcontainers import SortedList

d = SortedList(key=lambda x: x[0])

d.add((1, "1"))
d.add((0, "0"))
d.add((12, "12"))
d.add((5, "5"))

d

SortedKeyList([(0, '0'), (1, '1'), (5, '5'), (12, '12')], key=<function <lambda> at 0x7904b4088680>)

In [ ]:
from omegaconf import DictConfig, OmegaConf
from hydra.utils import instantiate
import sys, os
from hydra import initialize, compose

repo_dir = os.path.dirname(os.path.abspath("./"))
if repo_dir not in sys.path:
    print(f'add repository dir: {repo_dir}')
    sys.path.append(repo_dir)


def load_config(overrides=None):
    # Initialize Hydra without decorator
    with initialize(version_base="1.3", config_path="../configs"):
        cfg = compose(
            config_name="training",
            overrides=overrides if overrides else []
        )
        # Resolve all interpolations
        OmegaConf.resolve(cfg)
        return cfg


In [ ]:

from rl.pqn import PQN
import torch 
from copy import deepcopy

cfg = load_config()
agent_config = cfg.algo
env_config = cfg.envs

writer = instantiate(cfg.logger.tensorboard)
torch.set_float32_matmul_precision('high')

agent = PQN(agent_config)

tokenizer = agent.critic.state_embed.tokenizer
env_1 = instantiate(env_config.env, embedder=agent.action_embed_target, embed_tokenizer=tokenizer)
env_2 = deepcopy(env_1)
env_3 = deepcopy(env_1)
# env_2 = instantiate(env_config.env, embedder=agent.action_embed_target, embed_tokenizer=tokenizer)
# env_3 = instantiate(env_config.env, embedder=agent.action_embed_target, embed_tokenizer=tokenizer)

In [3]:
from rl.text_env import ParallelTextEnv

par_env = ParallelTextEnv([env_1, env_2, env_3])

In [ ]:
s, s_batch = par_env.reset()
a_embeds = par_env.get_extra_embeds(agent.critic.state_embed)
agent.select_action_batch(s_batch, a_embeds)

In [ ]:
par_env.step([1,3,3])

In [ ]:
env = instantiate(config.env)
test_env = instantiate(config.test_env)

In [ ]:
env.reset()

In [1]:
from torchrl.envs import EnvBase
from collections import defaultdict
from typing import Optional
import numpy as np
import torch
from tensordict import TensorDict, TensorDictBase
from tensordict.nn import TensorDictModule
from torch import nn

from torchrl.data import Bounded, Composite, Unbounded, Categorical, NonTensorSpec
from torchrl.envs import (
    CatTensors,
    EnvBase,
    Transform,
    TransformedEnv,
    UnsqueezeTransform,
    ParallelEnv
)


class CounterEnv(EnvBase):
    def __init__(self, batch_size=None, device=None, **kwargs):
         super().__init__()
         self.observation_spec = Composite(
             text=NonTensorSpec(dtype=str)
         )
         self.action_spec = Categorical(100, device=device, dtype=torch.int32)
         self._allow_done_after_reset = False
         self.run_type_checks = False
         
         # done spec and reward spec are set automatically

    def gen_params(self, batch_size) -> TensorDictBase:
    
        if batch_size is None:
            batch_size = []
        td = TensorDict({
                "params": TensorDict({})
            })
        if batch_size:
            td = td.expand(batch_size).contiguous()
        return td

    def _set_seed(self, seed: Optional[int]):
        rng = torch.manual_seed(seed)
        self.rng = rng

    def _step(self, tensordict):
          print(tensordict["text"])
          print(tensordict["action"])
          return TensorDict({
             "text": list(map(lambda t, a: t + a, tensordict["text"], tensordict["action"])),
            #  "params": tensordict["params"],
             "reward": 1.0,
             "done": False,
          })
     
    def _reset(self, tensordict, **kwargs):
          
          if tensordict is None or tensordict.is_empty():
                tensordict = self.gen_params(self.batch_size)

          return TensorDict({
             "text": "0",
             "params": tensordict["params"]
          })

In [ ]:
env = CounterEnv()
s = env.reset()
s = env.step(TensorDict({**s, "action": "2"}))["next"]
s = env.step(TensorDict({**s, "action": "2"}))["next"]
s = env.step(TensorDict({**s, "action": "2"}))["next"]
s["text"]

In [ ]:
def env_make():
    return CounterEnv()

parallel_env = ParallelEnv(3, env_make) 
s = parallel_env.reset()
s = env.step(TensorDict({**s, "action": ["2", "2", "2"]}))["next"]
s["text"]

In [ ]:
import sys
import os

%load_ext autoreload
%autoreload 2

repo_dir = os.path.dirname(os.path.abspath("./"))
if repo_dir not in sys.path:
    print(f'add repository dir: {repo_dir}')
    sys.path.append(repo_dir)


from rl.retrieval_babilong import RetrNoiseInjectionDataset, RetrSentenceSampler
from babilong_utils import TaskDataset

import datasets
from datasets import Dataset, load_dataset, load_from_disk
import torch
import sys
import time
import numpy as np
from collections import deque

# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


max_steps = 6
num_sentences = 50
task = "qa2_two-supporting-facts"
noise_train_path = "/home/nazar/pg19-train-with-sentences"
noise_path_test = "/home/nazar/pg19-test-with-sentences"
facts_train_path = f"/home/nazar/tasks_1-20_v1-2/en-10k/{task}_train.txt"
facts_test_path = f"/home/nazar/tasks_1-20_v1-2/en-10k/{task}_test.txt"


fact_dataset = TaskDataset(facts_train_path)  # max_n_facts=10)

noise_dataset = datasets.load_from_disk(noise_path_test)
noise_sampler = RetrSentenceSampler(noise_dataset)

dataset = RetrNoiseInjectionDataset(
    task_dataset=fact_dataset,
    noise_sentence_sampler=noise_sampler,
    num_sentences=num_sentences
)

In [5]:
from rl.babilong_env import BabilongEnv
# from rl.sacd import SAC, SACArgs
from rl.sarsa import SARSA, SARSAArgs
from rl.text_env import TextReplayBuffer
from transformers import AutoModel, AutoTokenizer
from rl.bert_predictor import BertPredictor
from rl.text_env import TextEnv, TextReplayBuffer


bert_name = "haisongzhang/roberta-tiny-cased"
tokenizer = AutoTokenizer.from_pretrained(bert_name, use_fast=True, revision="main")
bert_model = AutoModel.from_pretrained(bert_name, revision="main")


action_embed = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()
action_embed_target = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()

state_embed = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()
state_embed_target = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()

agent = SARSA(
    state_embed, action_embed, state_embed_target, action_embed_target, 
    SARSAArgs(gamma=0.95, tau=0.005,  lr=3e-4)
)


env = BabilongEnv( 
    embedder=agent.action_embed_target, 
    embed_tokenizer=tokenizer, 
    dataset=dataset,
    max_steps=max_steps
)

buffer = TextReplayBuffer(100_000, tokenizer = tokenizer)

In [ ]:
from tqdm import tqdm
import torch

learning_start = 10_000

step = 0
R = 0
for ep_number in tqdm(range(100_000)):

    s = env.reset()
    done = False
    a_embeds = env.get_extra_embeds(agent.critic.action_embed)
    agent.policy.update(agent.critic)

    qf_loss, alpha_loss = 0, 0

    a_all = []

    while not done:
        step += 1
        
        action = agent.select_action(s, a_embeds, random = step < learning_start or step % 10 == 0)
        s_next, a_data, reward, done = env.step(action)
        buffer.add(s, a_data, s_next, reward, done, 0)
        s = s_next
        R += reward
        a_all.append(action)
        
        if step > learning_start and step % 5 == 0:
            s_batch, a_batch, next_s_batch, r_batch, not_done_batch, entropy_batch = buffer.sample(32)
            qf_loss = agent.update(s_batch, a_batch, next_s_batch, r_batch, not_done_batch, step)
    
    if ep_number % 200 == 0 and ep_number > 0:
        print(R / 200, qf_loss)
        print(a_all, env.ref_ids)
        R = 0